# What does a short particle track reveal?
A marker that wanders freely can look directed for a few frames. A confined marker may look stationary when observation noise is large. Can we tell these cases apart using coordinates alone?

We compare a constant prediction with a small random forest on a held-out part of training. We then refit on all training tracks and write one label per test track. Only the evaluator gets test answers.

In [ ]:
import csv
from collections import defaultdict
from pathlib import Path

import numpy as np
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split

PUBLIC = Path("work/public")

def load_tracks(path):
    tracks = defaultdict(list)
    with path.open(newline="") as file:
        for row in csv.DictReader(file):
            tracks[row["track_id"]].append((int(row["frame"]), float(row["x"]), float(row["y"])))
    return {key: np.array(sorted(frames))[:, 1:] for key, frames in tracks.items()}

train_tracks = load_tracks(PUBLIC / "train_tracks.csv")
with (PUBLIC / "train_labels.csv").open(newline="") as file:
    labels = {r["track_id"]: r["label"] for r in csv.DictReader(file)}
print(f"{len(train_tracks)} labeled training tracks")

## Measure the shape of each path
The initial camera position and orientation tell us nothing about the motion. We use distances and angles between steps instead. A track with persistent drift tends to cover more distance over longer intervals. A confined track tends to revisit the same region. All features scale with the particle's typical step size.

In [ ]:
def features(points):
    steps = np.diff(points, axis=0)
    energy = np.mean(np.sum(steps**2, axis=1))
    # Average squared distance at different time intervals, relative to a single step.
    spreads = []
    for lag in (2, 4, 8, 16):
        offsets = points[lag:] - points[:-lag]
        spreads.append(np.mean(np.sum(offsets**2, axis=1)) / (lag * energy))
    alignment = np.mean(np.sum(steps[:-1] * steps[1:], axis=1)) / energy
    centered = points - points.mean(axis=0)
    # Dominant axis share: near one for an almost straight path.
    axes = np.linalg.eigvalsh(centered.T @ centered)
    axis_share = axes[-1] / axes.sum()
    net = np.sum((points[-1] - points[0])**2) / ((len(points) - 1) * energy)
    return [*spreads, alignment, axis_share, net]

ids = sorted(train_tracks)
X = np.array([features(train_tracks[key]) for key in ids])
y = np.array([labels[key] for key in ids])

## Compare against a constant prediction
Each track stays in one split. The reference model sees only shape measurements; IDs, class order, camera positions and the evaluator's noise labels are excluded. Macro F1 averages the three class F1 scores so one easy class cannot dominate the result.

In [ ]:
fit, valid = train_test_split(np.arange(len(y)), test_size=0.2, random_state=19, stratify=y)
constant = DummyClassifier(strategy="most_frequent").fit(X[fit], y[fit])
model = RandomForestClassifier(n_estimators=160, min_samples_leaf=3, random_state=19, n_jobs=1)
model.fit(X[fit], y[fit])
valid_pred = model.predict(X[valid])
print(f"Constant prediction: {f1_score(y[valid], constant.predict(X[valid]), average='macro'):.3f} macro F1")
print(f"Random forest:       {f1_score(y[valid], valid_pred, average='macro'):.3f} macro F1")
print(classification_report(y[valid], valid_pred, digits=3))

## Write the submission
Refit the same model on all labeled tracks. The CSV has exactly the two required columns. Validation above uses training labels only; the external grader measures the final test score.

In [ ]:
test_tracks = load_tracks(PUBLIC / "test_tracks.csv")
test_ids = sorted(test_tracks)
model.fit(X, y)
predictions = model.predict([features(test_tracks[key]) for key in test_ids])
with Path("work/submission.csv").open("w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["track_id", "label"])
    writer.writerows(zip(test_ids, predictions))
print(f"Wrote {len(test_ids)} predictions to work/submission.csv")